# Clean the Seattle-area restaurant data

Reads the output of `filter_seattle.ipynb` and writes two clean tables:
- `data/restaurants.csv`: one row per open restaurant
- `data/reviews.parquet`: one row per review that has text

Load them with `load_data.py` (`load_restaurants()`, `load_reviews()`) so the types come out right.

Requires `pip install pyarrow` for Parquet.

In [1]:
import json
import pandas as pd

## 1. Field completeness

In [2]:
restaurants = pd.read_json("data/meta-SeattleArea.json", lines=True, dtype={"gmap_id": str}, convert_dates=False)

fields = ["price", "category", "avg_rating", "num_of_reviews", "latitude", "longitude", "hours", "description"]
completeness = pd.DataFrame({"missing": [restaurants[f].isna().sum() for f in fields]}, index=fields)
completeness["missing %"] = (completeness["missing"] / len(restaurants) * 100).round(1)
print(f"{len(restaurants):,} restaurants before cleaning")
completeness

5,816 restaurants before cleaning


,missing,missing %
price,1514,26.0
category,0,0.0
avg_rating,0,0.0
num_of_reviews,0,0.0
latitude,0,0.0
longitude,0,0.0
hours,350,6.0
description,1783,30.7


## 2. Clean restaurants
- Drop permanently closed restaurants (keep temporarily closed, flagged).
- Price: some use `₩` instead of `$`; convert to a number 1–4, blank if unknown.
- Category: list → `|`-separated string so it fits in a CSV.
- MISC → `misc`, stored as a JSON string (e.g. `{"Atmosphere": ["Casual", "Cozy"], "Crowd": ["Locals"]}`). Useful for ambience terms and hole-in-the-wall signals like `Crowd: Locals` and `Payments: Cash-only`.
- Drop columns nobody needs (`state`, `hours`, `url`, `relative_results`).

In [3]:
restaurants = restaurants[restaurants["state"] != "Permanently closed"].drop_duplicates("gmap_id").copy()

restaurants["temporarily_closed"] = restaurants["state"] == "Temporarily closed"
restaurants["price"] = restaurants["price"].str.replace("₩", "$").str.len().astype("Int64")
restaurants["category"] = restaurants["category"].apply("|".join)
restaurants["misc"] = restaurants["MISC"].apply(json.dumps)

restaurants = restaurants[["gmap_id", "name", "address", "city", "latitude", "longitude", "category",
                           "price", "avg_rating", "num_of_reviews", "description", "misc", "temporarily_closed"]]

restaurants.to_csv("data/restaurants.csv", index=False)
print(f"{len(restaurants):,} restaurants after cleaning")
print(restaurants["price"].value_counts(dropna=False).sort_index())
restaurants.head()

4,931 restaurants after cleaning
price
1       1676
2       1940
3        151
4         17
<NA>    1147
Name: count, dtype: Int64


,gmap_id,name,address,city,latitude,longitude,category,price,avg_rating,num_of_reviews,description,misc,temporarily_closed
4,0x549015dcda35c8b5:0x59acf0fabf61e550,Citizen Campfire Beer Garden,"Citizen Campfire Beer Garden, 706 Taylor Ave N...",Seattle,47.625712,-122.345954,Bar & grill|Bar|Restaurant,<NA>,4.0,7,None,"{""Service options"": [""Curbside pickup"", ""No-co...",False
7,0x54906b5052cf8c7d:0xf5182d591ee7748,Hansen Central,"Hansen Central, 1526 13th Ave S #1, Seattle, W...",Seattle,47.589167,-122.315659,Restaurant,<NA>,5.0,2,None,"{""Service options"": [""Delivery""]}",False
8,0x5490438bd7dac025:0x21eeead6260273b4,Tacos el encanto,"Tacos el encanto, Tukwila, WA 98168",Tukwila,47.473748,-122.285002,Restaurant,<NA>,3.2,6,None,"{""Service options"": [""Takeout"", ""Dine-in"", ""De...",False
10,0x549015417a8f971d:0xfeeed6ecbc43eea1,Boca Restobar and Grill,"Boca Restobar and Grill, 2201 1st Ave, Seattle...",Seattle,47.612557,-122.345584,Argentinian restaurant,<NA>,4.4,28,None,"{""Service options"": [""Takeout"", ""Dine-in"", ""De...",False
11,0x54906b3060455059:0x884950a770b3787b,Olympic Bar,"Olympic Bar, 411 University St, Seattle, WA 98101",Seattle,47.607996,-122.333807,Restaurant|Bar,<NA>,4.6,15,None,"{""Service options"": [""Takeout"", ""Dine-in"", ""De...",False


## 3. Clean reviews
- Keep only reviews of the restaurants above (drops reviews of closed ones).
- Keep only reviews with text.
- Drop duplicates (same user, restaurant, and time).
- Business details (name, location, …) live in `restaurants.csv`; join on `gmap_id`.

In [4]:
ids = set(restaurants["gmap_id"])
keep = ["gmap_id", "user_id", "time", "rating", "text"]

chunks = pd.read_json("data/review-SeattleArea.json", lines=True, chunksize=500_000,
                      dtype={"user_id": str, "gmap_id": str}, convert_dates=False)
reviews = pd.concat(c.loc[c["gmap_id"].isin(ids), keep] for c in chunks)
n_open = len(reviews)

reviews = reviews[reviews["text"].notna() & (reviews["text"].str.strip() != "")]
n_text = len(reviews)

reviews = reviews.drop_duplicates(["user_id", "gmap_id", "time"])
reviews["time"] = pd.to_datetime(reviews["time"], unit="ms")
reviews["rating"] = reviews["rating"].astype("Int64")

reviews.to_parquet("data/reviews.parquet", index=False)
print(f"{n_open:,} reviews of open restaurants")
print(f"{n_text:,} with text")
print(f"{len(reviews):,} after removing duplicates")
reviews.head()

2,056,626 reviews of open restaurants
1,117,487 with text
1,089,428 after removing duplicates


,gmap_id,user_id,time,rating,text
74,0x549015dcda35c8b5:0x59acf0fabf61e550,102172916892203956834,2021-08-14 06:25:37.391,4,This place has a nice ambience. We tried the m...
75,0x549015dcda35c8b5:0x59acf0fabf61e550,111339593498069638516,2021-09-06 00:21:29.980,5,As much as I don't want to make this place mor...
76,0x549015dcda35c8b5:0x59acf0fabf61e550,105158290017774926865,2021-07-24 22:30:34.474,5,Love this place - adore the chill campfire vibes!
77,0x549015dcda35c8b5:0x59acf0fabf61e550,109265099310689872759,2021-08-21 03:06:12.546,2,Stars for the vibes only. Beer was undrinkable...
105,0x54906b5052cf8c7d:0xf5182d591ee7748,101250478423909853632,2018-06-18 02:25:06.929,5,Great ambiance but the host is a little odd. 😆
